Data Description:

Unique ID of each anime.

Anime title.

Anime broadcast type, such as TV, OVA, etc.
anime genre.

The number of episodes of each anime.

The average rating for each anime compared to the number of users who gave ratings.



Number of community members for each anime.

Objective:

The objective of this assignment is to implement a recommendation system using cosine similarity on an anime dataset.

Dataset:

Use the Anime Dataset which contains information about various anime, including their titles, genres,No.of episodes and user ratings etc.


Tasks:



Feature Extraction:

Decide on the features that will be used for computing similarity (e.g., genres, user ratings).

Convert categorical features into numerical representations if necessary.

Normalize numerical features if required.


Recommendation System:

Design a function to recommend anime based on cosine similarity.

Given a target anime, recommend a list of similar anime based on cosine similarity scores.

Experiment with different threshold values for similarity scores to adjust the recommendation list size.

Analyze the performance of the recommendation system and identify areas of improvement.



In [30]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import MinMaxScaler

In [31]:
df = pd.read_csv("anime.csv")

In [32]:
print(df.head())

print(df.info())

print(df.shape)

print(df.isnull().sum())

   anime_id                              name  \
0     32281                    Kimi no Na wa.   
1      5114  Fullmetal Alchemist: Brotherhood   
2     28977                          Gintama°   
3      9253                       Steins;Gate   
4      9969                     Gintama&#039;   

                                               genre   type episodes  rating  \
0               Drama, Romance, School, Supernatural  Movie        1    9.37   
1  Action, Adventure, Drama, Fantasy, Magic, Mili...     TV       64    9.26   
2  Action, Comedy, Historical, Parody, Samurai, S...     TV       51    9.25   
3                                   Sci-Fi, Thriller     TV       24    9.17   
4  Action, Comedy, Historical, Parody, Samurai, S...     TV       51    9.16   

   members  
0   200630  
1   793665  
2   114262  
3   673572  
4   151266  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12294 entries, 0 to 12293
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  

Data Preprocessing:

Load the dataset into a suitable data structure (e.g.,
pandas DataFrame).

Handle missing values, if any.

Explore the dataset to understand its structure and attributes.

In [33]:
# Handle missing values

df['genre'] = df['genre'].fillna('Unknown')

df['type'] = df['type'].fillna('Unknown')

df['rating'] = df['rating'].fillna(df['rating'].mean())


Convert Episodes Column into Numeric

In [34]:
# episodes column is currently object type.

# Replace Unknown values if present
df['episodes'] = df['episodes'].replace('Unknown', 0)

# Convert to numeric
df['episodes'] = pd.to_numeric(df['episodes'])


In [35]:
#Verify Data Types
print(df.dtypes)


anime_id      int64
name         object
genre        object
type         object
episodes      int64
rating      float64
members       int64
dtype: object


Genre Vectorization

In [36]:
cv = CountVectorizer(stop_words='english')

genre_matrix = cv.fit_transform(df['genre'])

print(genre_matrix.shape)

(12294, 47)


Normalize Numerical Features

In [37]:
scaler = MinMaxScaler()

numerical_features = scaler.fit_transform(df[['episodes', 'rating', 'members']])

print(numerical_features.shape)


(12294, 3)


Combine Features

In [38]:
from scipy.sparse import hstack

combined_features = hstack([genre_matrix, numerical_features])

print(combined_features.shape)


(12294, 50)


Compute Cosine Similarity

In [39]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(combined_features)

print(cosine_sim.shape)


(12294, 12294)


#Recommendation Function

In [40]:
def recommend_anime(anime_name, top_n=5):

    # Find anime index
    idx = df[df['name'].str.lower() == anime_name.lower()].index

    if len(idx) == 0:
        return "Anime not found"

    idx = idx[0]

    # Similarity scores
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort scores
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Remove same anime
    sim_scores = sim_scores[1:top_n+1]

    # Anime indices
    anime_indices = [i[0] for i in sim_scores]

    return df[['name', 'genre', 'rating']].iloc[anime_indices]

#Test Recommendation System

In [41]:
recommend_anime("One Piece")

,name,genre,rating
241,One Piece: Episode of Nami - Koukaishi no Nami...,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",8.27
231,One Piece: Episode of Merry - Mou Hitori no Na...,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",8.29
896,One Piece: Episode of Sabo - 3 Kyoudai no Kizu...,"Action, Adventure, Comedy, Drama, Fantasy, Sho...",7.78
352,One Piece Film: Strong World Episode 0,"Action, Adventure, Comedy, Fantasy, Shounen, S...",8.16
941,One Piece Movie 4: Dead End no Bouken,"Action, Adventure, Comedy, Fantasy, Shounen, S...",7.76


In [42]:
#Trying More Anime
recommend_anime("Naruto")

recommend_anime("Hunter x Hunter")

,name,genre,rating
146,Hunter x Hunter: Greed Island Final,"Action, Adventure, Shounen, Super Power",8.41
202,Hunter x Hunter: Greed Island,"Action, Adventure, Shounen, Super Power",8.33
145,Hunter x Hunter OVA,"Action, Adventure, Shounen, Super Power",8.41
1974,Hunter x Hunter Movie: Phantom Rouge,"Action, Adventure, Shounen, Super Power",7.39
2108,Hunter x Hunter Movie: The Last Mission,"Action, Adventure, Shounen, Super Power",7.35


Performance Analysis

Advantages

Fast and simple recommendation system

Works well for genre-based similarity

No user history required

Limitations

Recommendations may become repetitive

Does not consider user preferences

Genre-only similarity may not always be accurate

Improvements

Use TF-IDF Vectorizer

Add collaborative filtering

Build hybrid recommendation system

Include user ratings matrix


Interview Questions:

1. Can you explain the difference between user-based and item-based collaborative filtering?


User-Based Collaborative Filtering -->	Item-Based Collaborative Filtering

Similar users are identified -->	Similar items are identified

Uses user-user similarity	--> Uses item-item similarity
Less scalable	--> More scalable

Recommendations based on similar users -->	Recommendations based on similar items



2. What is collaborative filtering, and how does it work?
It is a recommendation technique that predicts user interests using behavior patterns of users or items.

Types


User-Based Collaborative Filtering

Item-Based Collaborative Filtering

Working

Find similarity between users/items

Predict preferences

Recommend relevant items